# TCC2 — Dengue Forecasting: Data Pipeline (Distrito Federal)

Exploratory analysis of the medallion data pipeline (Bronze → Silver → Gold) for dengue notification and meteorological data.

**Authors:** Pedro Lucas Santana & Thiago Ribeiro Freitas  
**Program:** Software Engineering — Universidade de Brasilia (UnB)  
**Focus:** Distrito Federal (Brasilia, IBGE 5300108)  

---

**Data Architecture:** This project follows the **Medallion Architecture** (Bronze → Silver → Gold) to transform
 raw epidemiological and meteorological data into model-ready features.

| Layer | SINAN (Dengue) | INMET (Weather) |
|-------|---------------|----------------|
| **Bronze** | Case-level notifications from OpenDataSUS | Hourly station readings |
| **Silver** | Municipality-week aggregates (~100 cols) | Weekly station aggregates (~17 cols) |
| **Gold** | + engineered features (~141 cols) | + lag/rolling features (~36 cols) |
| **Integrated** | SINAN Gold ⋈ INMET Gold → 172 columns per municipality-week |
| **Model-Ready** | Train/Val/Test splits with regression and classification targets |

---
## 1. Environment Setup

Install dependencies and configure access to the private HuggingFace repository.

In [ ]:
%%capture
!pip install -q huggingface_hub datasets pyarrow pandas numpy matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

warnings.filterwarnings('ignore')

# -- Plot defaults ----------------------------------------------------------
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

PALETTE = sns.color_palette('tab10')
C_BLUE, C_ORANGE, C_GREEN, C_RED, C_PURPLE = PALETTE[:5]
C_BROWN, C_PINK, C_GRAY, C_OLIVE, C_CYAN = PALETTE[5:10]

# -- HuggingFace config -----------------------------------------------------
HF_REPO   = 'thiagorfreitas/dengue-tcc2-data'
IBGE_DF   = '5300108'
STATION   = 'A001'  # INMET station for Brasilia

# Try Kaggle Secrets first, then env variable
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', None)

if HF_TOKEN:
    print('HF_TOKEN loaded successfully.')
else:
    print('WARNING: HF_TOKEN not found. Private repo access will fail.')

# -- Output dir (Kaggle) ----------------------------------------------------
OUT_DIR = Path('/kaggle/working')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Target municipality: Brasilia/DF (IBGE {IBGE_DF})')
print(f'Weather station: {STATION} (1.18 km from Brasilia center)')

In [ ]:
def load_hf_parquet(path_in_repo, filters=None):
    """Download a single parquet file from HuggingFace and load as DataFrame."""
    local = hf_hub_download(
        repo_id=HF_REPO,
        filename=path_in_repo,
        repo_type='dataset',
        token=HF_TOKEN
    )
    df = pd.read_parquet(local, filters=filters)
    return df


def load_hf_partitioned(base_path, years, filename='part-000.parquet',
                        filters=None, auto_glob=False):
    """Load partitioned parquet files (year=YYYY/) from HuggingFace and concat."""
    frames = []
    for yr in years:
        if auto_glob:
            # For datasets with multiple parts per year, try part-00000 through part-00009
            for part_idx in range(10):
                part_name = f'part-{part_idx:05d}.parquet'
                path = f'{base_path}/year={yr}/{part_name}'
                try:
                    df = load_hf_parquet(path, filters=filters)
                    frames.append(df)
                except Exception:
                    break  # No more parts for this year
        else:
            path = f'{base_path}/year={yr}/{filename}'
            try:
                df = load_hf_parquet(path, filters=filters)
                frames.append(df)
            except Exception as e:
                print(f'  Skipped year={yr}: {e}')
    if not frames:
        raise ValueError(f'No data loaded from {base_path}')
    return pd.concat(frames, ignore_index=True)


def save_fig(fig, name):
    """Save figure to Kaggle output directory."""
    fig.savefig(OUT_DIR / f'{name}.png', dpi=300, bbox_inches='tight')
    print(f'  Saved: {name}.png')


print('Helper functions ready.')

---
## 2. SINAN — Bronze Layer (Raw Case Data)

Bronze data was sourced from **OpenDataSUS** (SINAN-Net) as individual case-level CSV files.
 Due to privacy and size constraints, raw case data is not included in the repository.
 The Silver layer represents the first reproducible layer after aggregation to municipality-week level.

Each row in the raw SINAN data represents a single dengue notification, with fields such as:

| Field | Description |
|-------|------------|
| `DT_NOTIFIC` | Notification date |
| `SEM_NOT` | Epidemiological week of notification |
| `ID_MUNICIP` | IBGE municipality code |
| `DT_SIN_PRI` | Date of first symptoms |
| `CS_SEXO` | Sex (M/F/I) |
| `NU_IDADE_N` | Age (encoded) |
| `FEBRE` | Fever (1=yes, 2=no) |
| `MIALGIA` | Myalgia (1=yes, 2=no) |
| `CEFALEIA` | Headache (1=yes, 2=no) |
| `HOSPITALIZ` | Hospitalized (1=yes, 2=no) |
| `CLASSI_FIN` | Final classification (10=dengue, 11=severe dengue, ...) |
| `EVOLUCAO` | Outcome (1=cure, 2=death, ...) |
| ... | ~80 additional fields per notification |

**Volume:** Millions of individual notifications per year nationally.
 For Distrito Federal alone, tens of thousands of cases in epidemic years.

The transformation from Bronze to Silver involves:
1. Parsing dates and epidemiological weeks
2. Aggregating case counts by municipality and week
3. Computing proportions for symptoms, demographics, and severity indicators
4. Calculating derived metrics (mean age, notification delay, etc.)

---
## 3. SINAN — Silver Layer (Municipality-Week Aggregates)

The Silver layer aggregates individual notifications into **municipality-week** records.
 Each row represents one municipality in one epidemiological week, with ~100 columns
 covering notification counts, symptom proportions, demographic breakdowns, and severity indicators.

In [ ]:
# Load SINAN Silver — multiple years, filter to Distrito Federal
SINAN_SILVER_BASE = 'data/sinan/silver/sinan_tcc2_v2/official_observed'
YEARS_SILVER = list(range(2014, 2026))

print(f'Loading SINAN Silver for years {YEARS_SILVER[0]}-{YEARS_SILVER[-1]}...')
sinan_silver_raw = load_hf_partitioned(SINAN_SILVER_BASE, YEARS_SILVER)

# Ensure ibge_municipio is string for consistent filtering
sinan_silver_raw['ibge_municipio'] = sinan_silver_raw['ibge_municipio'].astype(str)
sinan_silver = sinan_silver_raw[sinan_silver_raw['ibge_municipio'] == IBGE_DF].copy()

# Parse date if available
if 'week_start' in sinan_silver.columns:
    sinan_silver['week_start'] = pd.to_datetime(sinan_silver['week_start'])
    sinan_silver = sinan_silver.sort_values('week_start').reset_index(drop=True)

print(f'\nNational rows loaded: {len(sinan_silver_raw):,}')
print(f'Distrito Federal rows: {len(sinan_silver):,}')
print(f'Shape: {sinan_silver.shape}')
print(f'Columns: {sinan_silver.shape[1]}')
print(f'Year range: {sinan_silver["ano"].min()} - {sinan_silver["ano"].max()}')

In [ ]:
sinan_silver.head(5)

In [ ]:
sinan_silver.describe().round(2)

In [ ]:
# Organize columns into logical groups
id_cols = ['ibge_municipio', 'municipio', 'uf', 'regiao', 'ano_semana',
           'ano', 'semana_epidemiologica', 'week_start']

count_cols = [c for c in sinan_silver.columns if c.startswith('qt_') or c == 'notificacoes']
prop_cols = [c for c in sinan_silver.columns if c.startswith('prop_')]
metric_cols = [c for c in sinan_silver.columns if 'media' in c.lower() or 'medio' in c.lower()]

other_cols = [c for c in sinan_silver.columns
              if c not in id_cols + count_cols + prop_cols + metric_cols]

summary = pd.DataFrame({
    'Group': ['Identifiers', 'Counts', 'Proportions', 'Metrics', 'Other'],
    'Count': [len([c for c in id_cols if c in sinan_silver.columns]),
              len(count_cols), len(prop_cols), len(metric_cols), len(other_cols)],
    'Examples': [
        ', '.join([c for c in id_cols if c in sinan_silver.columns][:4]),
        ', '.join(count_cols[:4]),
        ', '.join(prop_cols[:4]),
        ', '.join(metric_cols[:3]),
        ', '.join(other_cols[:3])
    ]
})
print(f'Total columns: {sinan_silver.shape[1]}')
summary

In [ ]:
# --- Plot: Weekly dengue notifications over time (full history) ---
fig, ax = plt.subplots(figsize=(14, 5))

ax.fill_between(sinan_silver['week_start'], sinan_silver['notificacoes'],
                alpha=0.3, color=C_BLUE)
ax.plot(sinan_silver['week_start'], sinan_silver['notificacoes'],
        linewidth=0.8, color=C_BLUE)

ax.set_title('Weekly Dengue Notifications — Distrito Federal', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Notifications')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Annotate peak
peak_idx = sinan_silver['notificacoes'].idxmax()
peak_date = sinan_silver.loc[peak_idx, 'week_start']
peak_val = sinan_silver.loc[peak_idx, 'notificacoes']
ax.annotate(f'Peak: {peak_val:,.0f}\n({peak_date.strftime("%b %Y")})',
            xy=(peak_date, peak_val),
            xytext=(30, 10), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color=C_RED),
            fontsize=10, color=C_RED, fontweight='bold')

plt.tight_layout()
save_fig(fig, 'sinan_silver_notifications_ts')
plt.show()

In [ ]:
# --- Plot: Distribution of weekly notification counts ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(sinan_silver['notificacoes'], bins=50, color=C_BLUE,
             edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution of Weekly Notifications', fontweight='bold')
axes[0].set_xlabel('Notifications per week')
axes[0].set_ylabel('Frequency')
axes[0].axvline(sinan_silver['notificacoes'].median(), color=C_RED,
                linestyle='--', label=f'Median: {sinan_silver["notificacoes"].median():.0f}')
axes[0].axvline(sinan_silver['notificacoes'].mean(), color=C_ORANGE,
                linestyle='--', label=f'Mean: {sinan_silver["notificacoes"].mean():.0f}')
axes[0].legend()

# Log scale
axes[1].hist(sinan_silver['notificacoes'].clip(lower=1), bins=50,
             color=C_GREEN, edgecolor='white', alpha=0.8)
axes[1].set_xscale('log')
axes[1].set_title('Distribution (log scale)', fontweight='bold')
axes[1].set_xlabel('Notifications per week (log)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
save_fig(fig, 'sinan_silver_notifications_dist')
plt.show()

print(f'Statistics:')
print(f'  Min:    {sinan_silver["notificacoes"].min():,.0f}')
print(f'  Median: {sinan_silver["notificacoes"].median():,.0f}')
print(f'  Mean:   {sinan_silver["notificacoes"].mean():,.0f}')
print(f'  Max:    {sinan_silver["notificacoes"].max():,.0f}')
print(f'  Std:    {sinan_silver["notificacoes"].std():,.0f}')

**Key observations — SINAN Silver:**
- ~100 columns per municipality-week, aggregated from individual case notifications
- Covers symptom proportions (`prop_febre`, `prop_mialgia`, ...), demographic splits, and severity counts
- Highly right-skewed distribution — most weeks have low counts, but epidemic peaks can reach thousands
- This is the first reproducible analytical layer in the pipeline

---
## 4. SINAN — Gold Layer (Engineered Features)

The Gold layer extends Silver with **engineered features** designed for time-series forecasting.
 These include composite health indices, seasonal encodings, lag features, rolling statistics,
 and binary alert labels.

In [ ]:
# Load SINAN Gold — partitioned, multiple parts per year
SINAN_GOLD_BASE = 'data/sinan/gold/sinan_tcc2_v2/official_dense'
YEARS_GOLD = list(range(2014, 2026))

print(f'Loading SINAN Gold for years {YEARS_GOLD[0]}-{YEARS_GOLD[-1]}...')
sinan_gold_raw = load_hf_partitioned(SINAN_GOLD_BASE, YEARS_GOLD, auto_glob=True)

sinan_gold_raw['ibge_municipio'] = sinan_gold_raw['ibge_municipio'].astype(str)
sinan_gold = sinan_gold_raw[sinan_gold_raw['ibge_municipio'] == IBGE_DF].copy()

if 'week_start' in sinan_gold.columns:
    sinan_gold['week_start'] = pd.to_datetime(sinan_gold['week_start'])
    sinan_gold = sinan_gold.sort_values('week_start').reset_index(drop=True)

print(f'\nNational rows loaded: {len(sinan_gold_raw):,}')
print(f'Distrito Federal rows: {len(sinan_gold):,}')
print(f'Shape: {sinan_gold.shape}')
print(f'Columns: {sinan_gold.shape[1]}')

# Identify new columns (Gold - Silver)
silver_cols = set(sinan_silver.columns)
gold_only_cols = [c for c in sinan_gold.columns if c not in silver_cols]
print(f'\nNew columns in Gold (not in Silver): {len(gold_only_cols)}')

In [ ]:
# Categorize the engineered features
indices = [c for c in gold_only_cols if c.startswith('indice_')]
seasonal = [c for c in gold_only_cols if 'sin' in c or 'cos' in c]
lags = [c for c in gold_only_cols if '_lag_' in c]
rolling = [c for c in gold_only_cols if 'movel' in c or 'rolling' in c.lower()]
diffs = [c for c in gold_only_cols if 'diff' in c or 'pct_change' in c or 'aceleracao' in c]
ratios = [c for c in gold_only_cols if 'razao' in c]
labels = [c for c in gold_only_cols if c.startswith('label_')]
other_new = [c for c in gold_only_cols
             if c not in indices + seasonal + lags + rolling + diffs + ratios + labels]

feat_groups = pd.DataFrame({
    'Feature Group': ['Composite Indices', 'Seasonal Encoding', 'Lag Features',
                      'Rolling Statistics', 'Differences / Acceleration',
                      'Ratios', 'Alert Labels', 'Other'],
    'Count': [len(indices), len(seasonal), len(lags), len(rolling),
              len(diffs), len(ratios), len(labels), len(other_new)],
    'Columns': [
        ', '.join(indices),
        ', '.join(seasonal),
        ', '.join(lags),
        ', '.join(rolling[:5]) + ('...' if len(rolling) > 5 else ''),
        ', '.join(diffs),
        ', '.join(ratios),
        ', '.join(labels),
        ', '.join(other_new[:5]) + ('...' if len(other_new) > 5 else '')
    ]
})
feat_groups

In [ ]:
# --- Plot: Correlation of key features with notificacoes ---
feature_cols = indices + lags + rolling + diffs + ratios
feature_cols = [c for c in feature_cols if c in sinan_gold.columns]

# Compute correlations with notification count
if 'notificacoes' in sinan_gold.columns and feature_cols:
    corr_target = sinan_gold[feature_cols + ['notificacoes']].corr()['notificacoes'].drop('notificacoes')
    corr_target = corr_target.dropna().sort_values(ascending=False)

    # Top 20 absolute correlations
    top_corr = corr_target.abs().sort_values(ascending=False).head(20)
    top_names = top_corr.index.tolist()

    fig, ax = plt.subplots(figsize=(10, 8))
    colors = [C_BLUE if corr_target[n] >= 0 else C_RED for n in top_names]
    ax.barh(range(len(top_names)), [corr_target[n] for n in top_names],
            color=colors, edgecolor='white')
    ax.set_yticks(range(len(top_names)))
    ax.set_yticklabels(top_names)
    ax.set_xlabel('Pearson Correlation with notificacoes')
    ax.set_title('Top 20 Engineered Features — Correlation with Notification Count',
                 fontweight='bold')
    ax.invert_yaxis()
    ax.axvline(0, color='black', linewidth=0.5)

    plt.tight_layout()
    save_fig(fig, 'sinan_gold_correlation_bars')
    plt.show()

In [ ]:
# --- Plot: Lag features tracking the notification signal ---
lag_cols_plot = [c for c in sinan_gold.columns if c.startswith('notificacoes_lag_')]
lag_cols_plot = sorted(lag_cols_plot,
                       key=lambda x: int(x.split('_')[-1]) if x.split('_')[-1].isdigit() else 99)

if lag_cols_plot and 'week_start' in sinan_gold.columns:
    fig, ax = plt.subplots(figsize=(14, 6))

    ax.plot(sinan_gold['week_start'], sinan_gold['notificacoes'],
            color='black', linewidth=1.5, label='notificacoes (current)', zorder=5)

    for i, col in enumerate(lag_cols_plot[:4]):
        lag_num = col.split('_')[-1]
        ax.plot(sinan_gold['week_start'], sinan_gold[col],
                linewidth=0.9, alpha=0.6, color=PALETTE[i+1],
                label=f'lag {lag_num} weeks')

    ax.set_title('Notification Lag Features Over Time — Distrito Federal', fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Notifications')
    ax.legend(loc='upper left')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

    plt.tight_layout()
    save_fig(fig, 'sinan_gold_lag_features')
    plt.show()

In [ ]:
# --- Heatmap: Correlation matrix of composite indices + key features ---
heatmap_cols = ['notificacoes'] + indices
heatmap_cols = [c for c in heatmap_cols if c in sinan_gold.columns]

if len(heatmap_cols) > 2:
    corr_matrix = sinan_gold[heatmap_cols].corr()

    fig, ax = plt.subplots(figsize=(10, 8))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
                cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                square=True, ax=ax, linewidths=0.5)
    ax.set_title('Correlation Matrix — Composite Indices vs Notifications',
                 fontweight='bold')

    plt.tight_layout()
    save_fig(fig, 'sinan_gold_corr_heatmap')
    plt.show()

**Key observations — SINAN Gold:**
- 141 columns: all Silver columns plus ~41 engineered features
- Composite indices capture multi-dimensional aspects of disease severity
- Lag features (1-12 weeks) provide the autoregressive signal essential for forecasting
- Alert labels (quantile-based) define classification targets for outbreak detection

---
## 5. INMET — Bronze Layer (Hourly Station Data)

The INMET Bronze layer contains **hourly meteorological readings** from automatic weather stations
 across Brazil. For Distrito Federal, we use station **A001** (Brasilia),
 located approximately 1.18 km from the city center.

Each record captures temperature, rainfall, humidity, pressure, wind speed, and solar radiation
 at hourly resolution — approximately **8,760 readings per station per year**.

In [ ]:
# Load INMET Bronze — one year for exploration
INMET_YEAR = 2023

print(f'Loading INMET Bronze (hourly) for {INMET_YEAR}...')
inmet_bronze_path = f'data/inmet/bronze/hourly/year={INMET_YEAR}/data.parquet'
inmet_bronze_raw = load_hf_parquet(inmet_bronze_path)

print(f'National rows: {len(inmet_bronze_raw):,}')
print(f'Columns: {inmet_bronze_raw.shape[1]}')
print(f'Stations: {inmet_bronze_raw["codigo_wmo"].nunique()}')

# Filter to Brasilia station A001
inmet_bronze_raw['codigo_wmo'] = inmet_bronze_raw['codigo_wmo'].astype(str)
inmet_bronze = inmet_bronze_raw[inmet_bronze_raw['codigo_wmo'] == STATION].copy()

# Parse date if needed
if 'data' in inmet_bronze.columns:
    inmet_bronze['data'] = pd.to_datetime(inmet_bronze['data'])

print(f'\nStation A001 (Brasilia) rows: {len(inmet_bronze):,}')
print(f'Shape: {inmet_bronze.shape}')

In [ ]:
inmet_bronze.head(5)

In [ ]:
# --- Plot: Hourly temperature and rainfall for the year ---
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Temperature
if 'temp_inst_c' in inmet_bronze.columns:
    axes[0].plot(inmet_bronze['data'], inmet_bronze['temp_inst_c'],
                 linewidth=0.3, alpha=0.6, color=C_RED)
    axes[0].set_ylabel('Temperature (C)')
    axes[0].set_title(f'Hourly Temperature — Station A001, Brasilia ({INMET_YEAR})',
                      fontweight='bold')

    # Add daily rolling mean
    daily_temp = inmet_bronze.set_index('data')['temp_inst_c'].resample('D').mean()
    axes[0].plot(daily_temp.index, daily_temp.values,
                 linewidth=1.2, color='darkred', label='Daily mean')
    axes[0].legend()

# Rainfall
if 'precipitacao_mm' in inmet_bronze.columns:
    axes[1].bar(inmet_bronze['data'], inmet_bronze['precipitacao_mm'].fillna(0),
                width=0.04, color=C_BLUE, alpha=0.6)
    axes[1].set_ylabel('Precipitation (mm)')
    axes[1].set_xlabel('Date')
    axes[1].set_title(f'Hourly Precipitation — Station A001, Brasilia ({INMET_YEAR})',
                      fontweight='bold')

plt.tight_layout()
save_fig(fig, 'inmet_bronze_hourly')
plt.show()

print(f'Note: ~3.9M rows/year nationally, hourly granularity.')
print(f'Station A001 is 1.18 km from Brasilia center.')

**Key observations — INMET Bronze:**
- Hourly readings from ~570 automatic stations across Brazil
- Station A001 provides continuous coverage for Distrito Federal
- Clear seasonal patterns: wet season (Oct-Mar) with higher rainfall and temperatures
- Dry season (Apr-Sep) with lower humidity and wider temperature ranges

---
## 6. INMET — Silver Layer (Weekly Station Aggregates)

The Silver layer aggregates hourly readings into **weekly summaries per station**,
 aligned to epidemiological weeks to match the SINAN temporal granularity.
 This includes computing means, extremes, rainfall counts, and data quality indicators.

In [ ]:
# Load INMET Silver for the same year
print(f'Loading INMET Silver for {INMET_YEAR}...')
inmet_silver_path = f'data/inmet/silver/weekly_stations_{INMET_YEAR}.parquet'
inmet_silver_raw = load_hf_parquet(inmet_silver_path)

inmet_silver_raw['codigo_wmo'] = inmet_silver_raw['codigo_wmo'].astype(str)
inmet_silver = inmet_silver_raw[inmet_silver_raw['codigo_wmo'] == STATION].copy()
inmet_silver = inmet_silver.sort_values('semana_epidemiologica').reset_index(drop=True)

print(f'National rows: {len(inmet_silver_raw):,}')
print(f'Station A001 rows: {len(inmet_silver):,}')
print(f'Shape: {inmet_silver.shape}')
print(f'Columns: {list(inmet_silver.columns)}')

In [ ]:
inmet_silver.head(10)

In [ ]:
# --- Plot: Weekly temperature and rainfall (Silver) ---
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

weeks = inmet_silver['semana_epidemiologica']

# Temperature
if 'temp_mean_c' in inmet_silver.columns:
    axes[0].plot(weeks, inmet_silver['temp_mean_c'],
                 color=C_RED, marker='o', markersize=3, label='Mean')
    if 'temp_min_c' in inmet_silver.columns and 'temp_max_c' in inmet_silver.columns:
        axes[0].fill_between(weeks, inmet_silver['temp_min_c'],
                             inmet_silver['temp_max_c'],
                             alpha=0.2, color=C_RED, label='Min-Max range')
    axes[0].set_ylabel('Temperature (C)')
    axes[0].set_title(f'Weekly Temperature — Station A001, Brasilia ({INMET_YEAR})',
                      fontweight='bold')
    axes[0].legend()

# Rainfall
if 'rain_sum_mm' in inmet_silver.columns:
    axes[1].bar(weeks, inmet_silver['rain_sum_mm'], color=C_BLUE, alpha=0.7)
    axes[1].set_ylabel('Rainfall (mm)')
    axes[1].set_xlabel('Epidemiological Week')
    axes[1].set_title(f'Weekly Rainfall — Station A001, Brasilia ({INMET_YEAR})',
                      fontweight='bold')

plt.tight_layout()
save_fig(fig, 'inmet_silver_weekly')
plt.show()

In [ ]:
# --- Data quality: coverage indicator ---
if 'n_valid_hours' in inmet_silver.columns:
    max_hours_per_week = 7 * 24  # 168
    inmet_silver['coverage_pct'] = (inmet_silver['n_valid_hours'] / max_hours_per_week * 100)

    fig, ax = plt.subplots(figsize=(12, 4))
    colors = [C_GREEN if cov >= 80 else (C_ORANGE if cov >= 50 else C_RED)
              for cov in inmet_silver['coverage_pct']]
    ax.bar(inmet_silver['semana_epidemiologica'], inmet_silver['coverage_pct'],
           color=colors, edgecolor='white')
    ax.axhline(80, color=C_GREEN, linestyle='--', alpha=0.5, label='80% threshold')
    ax.set_xlabel('Epidemiological Week')
    ax.set_ylabel('Data Coverage (%)')
    ax.set_title(f'Hourly Data Coverage per Week — Station A001 ({INMET_YEAR})',
                 fontweight='bold')
    ax.legend()
    ax.set_ylim(0, 105)

    low_cov = inmet_silver['low_coverage'].sum() if 'low_coverage' in inmet_silver.columns else 0
    print(f'Weeks with low coverage flag: {low_cov} / {len(inmet_silver)}')

    plt.tight_layout()
    save_fig(fig, 'inmet_silver_coverage')
    plt.show()

**Key observations — INMET Silver:**
- Hourly readings aggregated to weekly summaries aligned with epidemiological weeks
- 17 columns capturing temperature, rainfall, humidity, pressure, wind, and radiation
- Data quality tracked via `n_valid_hours` and `low_coverage` flag
- Ready for temporal joins with SINAN data

---
## 7. INMET — Gold Layer (Municipal Climate Features)

The Gold layer maps station data to municipalities and adds **lag and rolling features**
 for climate variables. Each station is associated with nearby municipalities,
 and temporal features capture delayed effects of weather on disease transmission.

In [ ]:
# Load INMET Gold — multiple years
print('Loading INMET Gold...')
inmet_gold_frames = []
for yr in range(2014, 2026):
    path = f'data/inmet/gold/weekly_municipal_climate_{yr}.parquet'
    try:
        df = load_hf_parquet(path)
        df['ibge_municipio'] = df['ibge_municipio'].astype(str)
        df_df = df[df['ibge_municipio'] == IBGE_DF]
        if len(df_df) > 0:
            inmet_gold_frames.append(df_df)
    except Exception as e:
        print(f'  Skipped {yr}: {e}')

inmet_gold = pd.concat(inmet_gold_frames, ignore_index=True)
inmet_gold = inmet_gold.sort_values(['ano_epi', 'semana_epidemiologica']).reset_index(drop=True)

print(f'\nDistrito Federal rows: {len(inmet_gold):,}')
print(f'Shape: {inmet_gold.shape}')
print(f'Columns ({inmet_gold.shape[1]}): {list(inmet_gold.columns)}')

In [ ]:
# Identify lag/rolling features added in Gold
silver_climate_cols = ['codigo_wmo', 'ano_epi', 'semana_epidemiologica',
                       'rain_sum_mm', 'rain_mean_mm', 'temp_mean_c', 'temp_min_c',
                       'temp_max_c', 'humidity_mean_pct', 'pressure_mean_mbar',
                       'wind_speed_mean_ms', 'radiation_mean_kj', 'n_valid_hours',
                       'rain_days', 'rain_heavy_days', 'temp_range_c', 'low_coverage']

gold_new = [c for c in inmet_gold.columns
            if c not in silver_climate_cols + ['ibge_municipio']]

lag_features = [c for c in gold_new if 'lag' in c.lower()]
rolling_features = [c for c in gold_new if 'mm4' in c.lower() or 'rolling' in c.lower()]
other_gold = [c for c in gold_new if c not in lag_features + rolling_features]

print(f'New lag features: {len(lag_features)}')
print(f'  {lag_features}')
print(f'\nNew rolling features: {len(rolling_features)}')
print(f'  {rolling_features}')
if other_gold:
    print(f'\nOther new: {other_gold}')

In [ ]:
# --- Plot: Climate features with lags overlaid ---
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

x = range(len(inmet_gold))
x_labels = [f"{int(r['ano_epi'])}-W{int(r['semana_epidemiologica']):02d}"
            for _, r in inmet_gold.iterrows()]

# Rainfall + lags
if 'rain_sum_mm' in inmet_gold.columns:
    axes[0].plot(x, inmet_gold['rain_sum_mm'], color=C_BLUE,
                 linewidth=1, label='rain_sum_mm (current)')
    for lag_c in ['rain_sum_mm_lag_1', 'rain_sum_mm_lag_4']:
        if lag_c in inmet_gold.columns:
            lag_n = lag_c.split('_')[-1]
            axes[0].plot(x, inmet_gold[lag_c], linewidth=0.7,
                         alpha=0.6, linestyle='--', label=f'lag {lag_n}w')
    axes[0].set_ylabel('Rainfall (mm)')
    axes[0].set_title('Rainfall with Lag Features — Distrito Federal', fontweight='bold')
    axes[0].legend(loc='upper right', fontsize=9)

# Temperature + lags
if 'temp_mean_c' in inmet_gold.columns:
    axes[1].plot(x, inmet_gold['temp_mean_c'], color=C_RED,
                 linewidth=1, label='temp_mean_c (current)')
    for lag_c in ['temp_mean_c_lag_1', 'temp_mean_c_lag_4']:
        if lag_c in inmet_gold.columns:
            lag_n = lag_c.split('_')[-1]
            axes[1].plot(x, inmet_gold[lag_c], linewidth=0.7,
                         alpha=0.6, linestyle='--', label=f'lag {lag_n}w')
    axes[1].set_ylabel('Temperature (C)')
    axes[1].set_title('Temperature with Lag Features — Distrito Federal', fontweight='bold')
    axes[1].legend(loc='upper right', fontsize=9)

# Humidity + lags
if 'humidity_mean_pct' in inmet_gold.columns:
    axes[2].plot(x, inmet_gold['humidity_mean_pct'], color=C_GREEN,
                 linewidth=1, label='humidity_mean_pct (current)')
    for lag_c in ['humidity_mean_pct_lag_1', 'humidity_mean_pct_lag_4']:
        if lag_c in inmet_gold.columns:
            lag_n = lag_c.split('_')[-1]
            axes[2].plot(x, inmet_gold[lag_c], linewidth=0.7,
                         alpha=0.6, linestyle='--', label=f'lag {lag_n}w')
    axes[2].set_ylabel('Humidity (%)')
    axes[2].set_xlabel('Week Index')
    axes[2].set_title('Humidity with Lag Features — Distrito Federal', fontweight='bold')
    axes[2].legend(loc='upper right', fontsize=9)

# Show only a subset of x-tick labels to avoid clutter
tick_step = max(1, len(x) // 15)
for ax in axes:
    ax.set_xticks(list(x)[::tick_step])
    ax.set_xticklabels([x_labels[i] for i in list(x)[::tick_step]],
                       rotation=45, ha='right', fontsize=8)

plt.tight_layout()
save_fig(fig, 'inmet_gold_climate_lags')
plt.show()

In [ ]:
# --- Table: Complete column listing for INMET Gold ---
col_descriptions = {
    'ibge_municipio': 'IBGE municipality code',
    'codigo_wmo': 'Weather station WMO code',
    'ano_epi': 'Epidemiological year',
    'semana_epidemiologica': 'Epidemiological week',
    'rain_sum_mm': 'Total weekly rainfall (mm)',
    'rain_mean_mm': 'Mean daily rainfall (mm)',
    'temp_mean_c': 'Mean weekly temperature (C)',
    'temp_min_c': 'Minimum temperature in week (C)',
    'temp_max_c': 'Maximum temperature in week (C)',
    'humidity_mean_pct': 'Mean relative humidity (%)',
    'pressure_mean_mbar': 'Mean atmospheric pressure (mbar)',
    'wind_speed_mean_ms': 'Mean wind speed (m/s)',
    'radiation_mean_kj': 'Mean solar radiation (kJ/m2)',
    'n_valid_hours': 'Valid hourly observations in week',
    'rain_days': 'Days with any rainfall',
    'rain_heavy_days': 'Days with heavy rainfall (>20mm)',
    'temp_range_c': 'Temperature range (max - min)',
    'low_coverage': 'Low data coverage flag',
}
# Add lag descriptions dynamically
for c in inmet_gold.columns:
    if c not in col_descriptions:
        if 'lag_' in c:
            base = c.rsplit('_lag_', 1)[0]
            lag_n = c.rsplit('_lag_', 1)[1]
            col_descriptions[c] = f'{base} lagged {lag_n} week(s)'
        elif '_mm4' in c:
            col_descriptions[c] = f'4-week rolling mean of {c.replace("_mm4", "")}'
        else:
            col_descriptions[c] = '—'

col_df = pd.DataFrame([
    {'Column': c, 'Description': col_descriptions.get(c, '—'),
     'Non-null': inmet_gold[c].notna().sum(),
     'Dtype': str(inmet_gold[c].dtype)}
    for c in inmet_gold.columns
])
col_df

**Key observations — INMET Gold:**
- 36 columns: station-level weekly summaries plus municipality mapping and temporal features
- Lag features capture delayed climate effects (1, 2, 4, 8 weeks) for rainfall, temperature, and humidity
- Rolling means smooth short-term noise for more stable predictors
- Ready for integration with SINAN Gold via `(ibge_municipio, ano_epi, semana_epidemiologica)`

---
## 8. Data Integration (SINAN Gold + INMET Gold)

The integrated dataset joins SINAN Gold (dengue features) with INMET Gold (climate features)
 on the composite key `(ibge_municipio, ano, semana_epidemiologica)`.

$$\text{Integrated} = \text{SINAN Gold} \bowtie_{\text{ibge, year, week}} \text{INMET Gold}$$

This produces a comprehensive feature set of **172 columns** per municipality-week.

In [ ]:
# Load integrated dataset — single file
print('Loading integrated dataset...')
integrated_raw = load_hf_parquet('data/integrated/sinan_inmet_municipal_weekly.parquet')
integrated_raw['ibge_municipio'] = integrated_raw['ibge_municipio'].astype(str)

integrated = integrated_raw[integrated_raw['ibge_municipio'] == IBGE_DF].copy()

print(f'National rows: {len(integrated_raw):,}')
print(f'Distrito Federal rows: {len(integrated):,}')
print(f'Shape: {integrated.shape}')
print(f'Total columns: {integrated.shape[1]}')

if 'week_start' in integrated.columns:
    integrated['week_start'] = pd.to_datetime(integrated['week_start'])
    integrated = integrated.sort_values('week_start').reset_index(drop=True)
    print(f'Date range: {integrated["week_start"].min()} to {integrated["week_start"].max()}')

In [ ]:
# --- Column count by source ---
sinan_exclusive = [c for c in integrated.columns
                   if c in sinan_gold.columns and c not in inmet_gold.columns]
inmet_exclusive = [c for c in integrated.columns
                   if c in inmet_gold.columns and c not in sinan_gold.columns]
shared_cols = [c for c in integrated.columns
               if c in sinan_gold.columns and c in inmet_gold.columns]
integration_only = [c for c in integrated.columns
                    if c not in sinan_gold.columns and c not in inmet_gold.columns]

source_summary = pd.DataFrame({
    'Source': ['SINAN-only', 'INMET-only', 'Shared keys', 'Integration-specific', 'Total'],
    'Column Count': [len(sinan_exclusive), len(inmet_exclusive),
                     len(shared_cols), len(integration_only), integrated.shape[1]],
    'Examples': [
        ', '.join(sinan_exclusive[:4]),
        ', '.join(inmet_exclusive[:4]),
        ', '.join(shared_cols[:4]),
        ', '.join(integration_only[:4]) if integration_only else '—',
        '—'
    ]
})
source_summary

In [ ]:
# --- Heatmap: Missing values by variable group over time ---
# Group columns by category
climate_cols_int = [c for c in integrated.columns
                    if any(k in c for k in ['rain', 'temp', 'humid', 'pressure',
                                            'wind', 'radiation', 'coverage'])]
sinan_count_cols = [c for c in integrated.columns
                    if c.startswith('qt_') or c == 'notificacoes']
sinan_prop_cols = [c for c in integrated.columns if c.startswith('prop_')]
sinan_lag_cols = [c for c in integrated.columns if 'notificacoes_lag' in c]
sinan_index_cols = [c for c in integrated.columns if c.startswith('indice_')]
label_cols_int = [c for c in integrated.columns if c.startswith('label_')]

groups = {
    'Climate vars': climate_cols_int,
    'Notification counts': sinan_count_cols,
    'Symptom proportions': sinan_prop_cols[:10],  # Sample for readability
    'Notification lags': sinan_lag_cols,
    'Composite indices': sinan_index_cols,
    'Alert labels': label_cols_int,
}

# Build year-based missing data matrix
if 'ano' in integrated.columns:
    years_avail = sorted(integrated['ano'].unique())
    missing_matrix = []
    group_labels = []

    for gname, gcols in groups.items():
        gcols_present = [c for c in gcols if c in integrated.columns]
        if gcols_present:
            row = []
            for yr in years_avail:
                mask = integrated['ano'] == yr
                yr_data = integrated.loc[mask, gcols_present]
                pct_missing = yr_data.isna().mean().mean() * 100
                row.append(pct_missing)
            missing_matrix.append(row)
            group_labels.append(f'{gname} ({len(gcols_present)} cols)')

    missing_df = pd.DataFrame(missing_matrix,
                              index=group_labels,
                              columns=[str(int(y)) for y in years_avail])

    fig, ax = plt.subplots(figsize=(14, 6))
    sns.heatmap(missing_df, annot=True, fmt='.1f', cmap='YlOrRd',
                vmin=0, vmax=100, ax=ax, linewidths=0.5,
                cbar_kws={'label': '% Missing'})
    ax.set_title('Missing Data by Variable Group and Year — Distrito Federal',
                 fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Variable Group')

    plt.tight_layout()
    save_fig(fig, 'integrated_missing_heatmap')
    plt.show()

In [ ]:
# --- Bar chart: Climate data coverage by year ---
if 'ano' in integrated.columns and climate_cols_int:
    coverage_by_year = []
    for yr in sorted(integrated['ano'].unique()):
        mask = integrated['ano'] == yr
        total_weeks = mask.sum()
        # A week has climate data if any climate column is non-null
        climate_present = [c for c in climate_cols_int if c in integrated.columns]
        if climate_present:
            has_climate = integrated.loc[mask, climate_present].notna().any(axis=1).sum()
        else:
            has_climate = 0
        coverage_by_year.append({
            'year': int(yr),
            'total_weeks': total_weeks,
            'with_climate': has_climate,
            'without_climate': total_weeks - has_climate
        })

    cov_df = pd.DataFrame(coverage_by_year)

    fig, ax = plt.subplots(figsize=(12, 5))
    x_pos = range(len(cov_df))
    ax.bar(x_pos, cov_df['with_climate'], label='With INMET data', color=C_GREEN, alpha=0.8)
    ax.bar(x_pos, cov_df['without_climate'], bottom=cov_df['with_climate'],
           label='Without INMET data', color=C_RED, alpha=0.6)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(cov_df['year'].astype(str), rotation=45)
    ax.set_xlabel('Year')
    ax.set_ylabel('Weeks')
    ax.set_title('Climate Data Coverage by Year — Distrito Federal', fontweight='bold')
    ax.legend()

    total_missing_pct = cov_df['without_climate'].sum() / cov_df['total_weeks'].sum() * 100
    ax.text(0.98, 0.95, f'Overall: {total_missing_pct:.1f}% missing climate data',
            transform=ax.transAxes, ha='right', va='top', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    plt.tight_layout()
    save_fig(fig, 'integrated_climate_coverage')
    plt.show()

In [ ]:
# --- Show one complete row transposed (all 172 columns) ---
if len(integrated) > 0:
    sample_idx = len(integrated) // 2  # Middle row
    sample_row = integrated.iloc[sample_idx]
    
    transposed = pd.DataFrame({
        'Column': sample_row.index,
        'Value': sample_row.values,
        'Dtype': [str(integrated[c].dtype) for c in sample_row.index]
    })
    
    print(f'Sample row #{sample_idx} — all {len(transposed)} columns:')
    # Show in compact form
    with pd.option_context('display.max_rows', 180, 'display.max_colwidth', 40):
        display(transposed)

**Key observations — Integrated Dataset:**
- 172 columns combining epidemiological and meteorological features
- Join on `(ibge_municipio, ano, semana_epidemiologica)` preserves temporal alignment
- Climate data is available for most weeks, with ~14.3% missing (station gaps)
- Station A001 provides coverage for all of Distrito Federal

---
## 9. Model-Ready Splits

The final stage produces **train/validation/test splits** with defined targets:

| Target | Type | Description |
|--------|------|------------|
| `notificacoes_t4` | Regression | Notification count 4 weeks ahead |
| `risco_surto_t4` | Classification | Outbreak risk 4 weeks ahead (0=low, 1=medium, 2=high, 3=outbreak) |

Splits are **temporal** — no data leakage across time:
- **Train:** years < 2022
- **Validation:** 2022-2023
- **Test:** 2024+

In [ ]:
# Load model-ready splits
print('Loading model-ready splits...')
splits = {}
for split_name in ['train', 'val', 'test']:
    path = f'data/model_ready/{split_name}.parquet'
    df = load_hf_parquet(path)
    df['ibge_municipio'] = df['ibge_municipio'].astype(str)
    df_df = df[df['ibge_municipio'] == IBGE_DF].copy()
    splits[split_name] = df_df
    print(f'  {split_name}: {len(df):,} national -> {len(df_df):,} DF, '
          f'{df_df.shape[1]} cols')

train_df = splits['train']
val_df = splits['val']
test_df = splits['test']

In [ ]:
# --- Table: Split summary ---
split_info = []
for name, sdf in splits.items():
    year_col = 'ano' if 'ano' in sdf.columns else None
    if year_col:
        yr_range = f'{int(sdf[year_col].min())} - {int(sdf[year_col].max())}'
    else:
        yr_range = '—'

    info = {
        'Split': name.capitalize(),
        'Rows (DF)': len(sdf),
        'Columns': sdf.shape[1],
        'Year Range': yr_range,
    }
    if 'notificacoes_t4' in sdf.columns:
        info['Target Mean'] = f'{sdf["notificacoes_t4"].mean():,.1f}'
        info['Target Max'] = f'{sdf["notificacoes_t4"].max():,.0f}'
    if 'risco_surto_t4' in sdf.columns:
        info['Classes'] = str(sorted(sdf['risco_surto_t4'].dropna().unique().tolist()))

    split_info.append(info)

pd.DataFrame(split_info)

In [ ]:
# --- Histogram: Target distribution per split (notificacoes_t4) ---
if all('notificacoes_t4' in splits[s].columns for s in splits):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

    colors = {'train': C_BLUE, 'val': C_ORANGE, 'test': C_GREEN}

    for ax, (name, sdf) in zip(axes, splits.items()):
        target = sdf['notificacoes_t4'].dropna()
        ax.hist(target, bins=40, color=colors[name], edgecolor='white', alpha=0.8)
        ax.set_title(f'{name.capitalize()} (n={len(target):,})', fontweight='bold')
        ax.set_xlabel('notificacoes_t4')
        ax.axvline(target.mean(), color=C_RED, linestyle='--',
                   label=f'Mean: {target.mean():,.0f}')
        ax.axvline(target.median(), color='black', linestyle=':',
                   label=f'Median: {target.median():,.0f}')
        ax.legend(fontsize=9)

    axes[0].set_ylabel('Frequency')
    fig.suptitle('Regression Target Distribution (notificacoes_t4) by Split',
                 fontweight='bold', fontsize=14, y=1.02)

    plt.tight_layout()
    save_fig(fig, 'model_ready_target_hist')
    plt.show()

In [ ]:
# --- Box plot: Target by split ---
if all('notificacoes_t4' in splits[s].columns for s in splits):
    fig, ax = plt.subplots(figsize=(10, 6))

    box_data = [splits[s]['notificacoes_t4'].dropna() for s in ['train', 'val', 'test']]
    bp = ax.boxplot(box_data, labels=['Train', 'Validation', 'Test'],
                    patch_artist=True, showfliers=True, flierprops={'markersize': 3})

    for patch, color in zip(bp['boxes'], [C_BLUE, C_ORANGE, C_GREEN]):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)

    ax.set_ylabel('notificacoes_t4')
    ax.set_title('Target Distribution by Split — Distrito Federal', fontweight='bold')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

    # Annotate max values
    for i, (name, sdf) in enumerate(splits.items()):
        max_val = sdf['notificacoes_t4'].max()
        ax.annotate(f'max: {max_val:,.0f}', xy=(i+1, max_val),
                    xytext=(15, 5), textcoords='offset points',
                    fontsize=9, color=C_RED, fontweight='bold')

    plt.tight_layout()
    save_fig(fig, 'model_ready_target_boxplot')
    plt.show()

In [ ]:
# --- Class distribution bar chart (risco_surto_t4) ---
if all('risco_surto_t4' in splits[s].columns for s in splits):
    class_labels = {0: 'Low (<=p50)', 1: 'Medium (p50-p75)',
                    2: 'High (p75-p90)', 3: 'Outbreak (>p90)'}

    fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
    colors_class = [C_GREEN, C_ORANGE, C_RED, C_PURPLE]

    for ax, (name, sdf) in zip(axes, splits.items()):
        counts = sdf['risco_surto_t4'].value_counts().sort_index()
        bars = ax.bar([class_labels.get(c, str(c)) for c in counts.index],
                      counts.values,
                      color=[colors_class[int(c)] if int(c) < len(colors_class)
                             else C_GRAY for c in counts.index],
                      edgecolor='white')

        # Add count labels on bars
        for bar, val in zip(bars, counts.values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    str(val), ha='center', va='bottom', fontsize=10, fontweight='bold')

        ax.set_title(f'{name.capitalize()} (n={len(sdf):,})', fontweight='bold')
        ax.set_xlabel('Risk Class')
        ax.tick_params(axis='x', rotation=30)

    axes[0].set_ylabel('Count')
    fig.suptitle('Classification Target Distribution (risco_surto_t4) by Split',
                 fontweight='bold', fontsize=14, y=1.02)

    plt.tight_layout()
    save_fig(fig, 'model_ready_class_dist')
    plt.show()

In [ ]:
# --- Time series: train/val/test regions with different colors ---
fig, ax = plt.subplots(figsize=(14, 6))

for name, sdf, color in [('Train', train_df, C_BLUE),
                          ('Validation', val_df, C_ORANGE),
                          ('Test', test_df, C_GREEN)]:
    if 'week_start' in sdf.columns and 'notificacoes_t4' in sdf.columns:
        sdf_sorted = sdf.sort_values('week_start')
        ax.fill_between(pd.to_datetime(sdf_sorted['week_start']),
                        sdf_sorted['notificacoes_t4'],
                        alpha=0.3, color=color)
        ax.plot(pd.to_datetime(sdf_sorted['week_start']),
                sdf_sorted['notificacoes_t4'],
                linewidth=0.9, color=color, label=name)

ax.set_title('Regression Target (notificacoes_t4) — Train / Validation / Test',
             fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Notifications (4-week ahead)')
ax.legend(loc='upper left', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Annotate distribution shift
train_max = train_df['notificacoes_t4'].max() if 'notificacoes_t4' in train_df.columns else 0
test_max = test_df['notificacoes_t4'].max() if 'notificacoes_t4' in test_df.columns else 0
ax.text(0.98, 0.95,
        f'Distribution shift: train max = {train_max:,.0f} vs test max = {test_max:,.0f}',
        transform=ax.transAxes, ha='right', va='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9),
        fontweight='bold')

plt.tight_layout()
save_fig(fig, 'model_ready_ts_splits')
plt.show()

**Key observations — Model-Ready Splits:**

- **Temporal splits** prevent data leakage: train (< 2022), validation (2022-2023), test (2024+)
- **Distribution shift:** the test set contains the unprecedented 2024 outbreak (max ~22,278 notifications)
  while training data peaks at ~3,968 — this is the central modeling challenge
- **Class imbalance:** outbreak class (3) is naturally rare, reflecting real epidemiological patterns
- **Dual targets:** regression (`notificacoes_t4`) for magnitude; classification (`risco_surto_t4`) for risk level

---
## 10. Summary

### Pipeline Overview

| Layer | SINAN Rows (DF) | SINAN Cols | INMET Rows (DF) | INMET Cols |
|-------|----------------|------------|-----------------|------------|
| Bronze | N/A (case-level, not stored) | ~80 | ~8,760/yr (station) | 22 |
| Silver | ~500+ | ~100 | ~52/yr (station) | 17 |
| Gold | ~500+ | ~141 | ~52/yr (municipality) | 36 |
| Integrated | ~1,395 | 172 | (included) | (included) |
| Model-Ready | ~1,391 total | 164 | (included) | (included) |

### Key Findings

1. **Data volume:** the medallion architecture reduces millions of hourly/case-level records
   into ~1,400 municipality-week observations for Distrito Federal, each described by 172 features.

2. **Feature richness:** composite severity indices, lag/rolling statistics, and seasonal encodings
   provide a comprehensive feature set for time-series forecasting.

3. **Climate integration:** INMET station A001 covers Distrito Federal with ~85.7% weekly coverage;
   lagged climate features capture the 1-8 week delay between weather conditions and disease incidence.

4. **Distribution shift:** the 2024 dengue outbreak in Distrito Federal produced notification counts
   ~5.6x higher than any previous peak — a critical challenge for model generalization.

5. **Temporal integrity:** train/validation/test splits are strictly temporal,
   ensuring no information leakage from future to past.

---
*Notebook generated for TCC2 defense — Software Engineering, Universidade de Brasilia.*